In [3]:
import pandas as pd

In [ ]:
df_app = pd.read_csv("dataset/application.csv")
df_p_app = pd.read_csv("dataset/p_application.csv")
df_joined = df_p_app.merge(df_app, on="curr_id", how="left")

In [ ]:
print("Shape of joined dataframe:", df_joined.shape)
# df_joined.head()
# df_joined.columns.tolist()

Shape of joined dataframe: (4999, 26)


In [6]:
selected_columns = [
    'sex', 'car', 'income', 'income_type', 'education',
    'family_status', 'house', 'organization', 'contract_x',
    'amount', 'down_payment', 'start_day_x', 'purpose',
    'contract_status', 'payment_type', 'reject_reason'
]

df_selected = df_joined[selected_columns].copy()
# df_selected

In [ ]:

df_filtered = df_selected.dropna(subset=['income', 'income_type', 'down_payment'])
df_filtered['label_refused'] = (df_selected['contract_status'] == 'Refused').astype(int)
print("Số dòng sau khi lọc:", df_filtered.shape[0])
# df_filtered.head()

In [8]:
# Lọc dữ liệu cho Logistic Regression
df_lr_full = df_filtered[['income', 'amount', 'down_payment', 'label_refused']].copy()

# Bước 1: Tạo tập huấn luyện 30 mẫu (15 mỗi nhãn, down_payment ≠ 0)
df_0_train = df_lr_full[(df_lr_full['label_refused'] == 0) & (df_lr_full['down_payment'].fillna(0) != 0)].sample(15, random_state=42)
df_1_train = df_lr_full[(df_lr_full['label_refused'] == 1) & (df_lr_full['down_payment'].fillna(0) != 0)].sample(15, random_state=42)
df_lr_short_train = pd.concat([df_0_train, df_1_train], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

# Bước 2: Tạo tập test 10 mẫu (5 mỗi nhãn, down_payment ≠ 0)
df_0_test = df_lr_full[(df_lr_full['label_refused'] == 0) & (df_lr_full['down_payment'].fillna(0) != 0)].drop(df_0_train.index).sample(5, random_state=1)
df_1_test = df_lr_full[(df_lr_full['label_refused'] == 1) & (df_lr_full['down_payment'].fillna(0) != 0)].drop(df_1_train.index).sample(5, random_state=1)
df_lr_short_test = pd.concat([df_0_test, df_1_test], ignore_index=True).sample(frac=1, random_state=1).reset_index(drop=True)

In [9]:
#  Lưu thành file Excel
with pd.ExcelWriter("Team11_HW6.xlsx", engine='openpyxl', mode='w') as writer:
    df_filtered.to_excel(writer, sheet_name='Data', index=False)
    df_lr_full.to_excel(writer, sheet_name='Data-LR', index=False)
    df_lr_short_train.to_excel(writer, sheet_name='Data-LR-Short-Train', index=False)
    df_lr_short_test.to_excel(writer, sheet_name='Data-LR-Short-Test', index=False)

print("Đã lưu file Team11_HW6.xlsx với 4 sheet.")


Đã lưu file Team11_HW6.xlsx với 4 sheet.


In [15]:
df_filtered.columns.tolist()

['sex',
 'car',
 'income',
 'income_type',
 'education',
 'family_status',
 'house',
 'organization',
 'contract_x',
 'amount',
 'down_payment',
 'start_day_x',
 'purpose',
 'contract_status',
 'payment_type',
 'reject_reason',
 'label_refused']

In [ ]:
# income amount down_payment label_refused